In [1]:
pip install pandas numpy rank-bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


***new code***

In [1]:
import re
import pandas as pd
from rank_bm25 import BM25Okapi

# ========= KONFIG =========
CSV_PATH = "../olahData/nondup_labeled_dataset.csv"
ENCODING = "utf-8"
TOPK = 20

TEXT_COLS = ["name", "category_breadcrumb", "shop_city"]

OUT_COLS = [
    "id", "name", "url",
    "category_breadcrumb",
    "price_number",          
    "discountPercentage",
    "ratingAverage",
    "countSold", "has_Promo", "umkm_label",
    "shop_id", "shop_name", "shop_city", "shop_tier"
]

# ========= UTIL =========
def safe_get_col(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df))

def basic_tokens(text: str):
    text = text.lower()
    # agar "kalung-cantik" dianggap pemisah -> "kalung cantik"
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    return text.split()

def tokenize_with_bigrams(text: str):
    """
    Tokenize unigram + bigram (phrase) untuk bantu pencarian frasa.
    Contoh: "kalung cantik" -> ["kalung","cantik","kalung_cantik"]
    """
    toks = basic_tokens(text)
    bigrams = [f"{toks[i]}_{toks[i+1]}" for i in range(len(toks)-1)]
    return toks + bigrams

def build_document(df: pd.DataFrame) -> pd.Series:
    parts = [safe_get_col(df, c) for c in TEXT_COLS]
    doc = parts[0]
    for p in parts[1:]:
        doc = doc + " " + p
    return doc

# ========= LOAD =========
df = pd.read_csv(CSV_PATH, encoding=ENCODING)

rename_map = {
    "shop.id": "shop_id",
    "shop.name": "shop_name",
    "shop.city": "shop_city",
    "shop.tier": "shop_tier",
    "category.breadcrumb": "category_breadcrumb",
    "price.number": "price_number",
    "price.discountPercentage": "discountPercentage",
}
for old, new in rename_map.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

# ========= BUILD CORPUS =========
df["doc"] = build_document(df)

# tokenisasi corpus: unigram + bigram
corpus_tokens = df["doc"].apply(tokenize_with_bigrams).tolist()
bm25 = BM25Okapi(corpus_tokens)

# ========= SEARCH =========
def bm25_search(query: str, topk: int = 20, require_all_terms: bool = True) -> pd.DataFrame:
    # token query
    q_unigrams = basic_tokens(query)
    q_tokens = tokenize_with_bigrams(query)

    scores = bm25.get_scores(q_tokens)

    # Ambil kandidat lebih banyak dulu, nanti kita filter/urutkan lagi
    candidate_k = max(topk * 10, 200)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:candidate_k]

    out = df.iloc[top_idx].copy()
    out["bm25_score"] = [scores[i] for i in top_idx]

    # (1) AND filter: wajib mengandung semua kata query (unigram)
    if require_all_terms and q_unigrams:
        doc_tokens_set = out["doc"].apply(lambda x: set(basic_tokens(x)))
        mask = doc_tokens_set.apply(lambda s: all(t in s for t in q_unigrams))
        out = out[mask]

    # (2) Bonus tambahan kalau frasa bigram ada persis di dokumen
    #     Ini membuat "kalung_cantik" jauh naik dibanding hanya "cantik"
    q_bigrams = [f"{q_unigrams[i]}_{q_unigrams[i+1]}" for i in range(len(q_unigrams)-1)]
    if q_bigrams:
        out["phrase_hit"] = out["doc"].apply(
            lambda x: sum(1 for bg in q_bigrams if bg in "_".join(basic_tokens(x)))  # cek urutan kata
        )
        out["bm25_score"] = out["bm25_score"] + (out["phrase_hit"] * 5.0)  # boost 5x, bisa kamu ubah

    # urutkan lagi setelah filter/boost, ambil topk
    out = out.sort_values("bm25_score", ascending=False).head(topk)

    cols = [c for c in OUT_COLS if c in out.columns]
    cols = cols + ["bm25_score"]
    return out[cols].reset_index(drop=True)

if __name__ == "__main__":
    print("BM25 siap. Contoh query:")
    q = input("Masukkan query (contoh: 'kalung cantik'): ").strip()
    res = bm25_search(q, topk=TOPK, require_all_terms=True)
    print(res.to_string(index=False))

    res.to_csv("bm25_results2.csv", index=False, encoding="utf-8-sig")
    print("\nDisimpan: bm25_results2.csv")


BM25 siap. Contoh query:
          id                                                                                                name                                                                                                                                                                                                                                                                             url                    category_breadcrumb  price_number  discountPercentage  ratingAverage  countSold umkm_label             shop_id               shop_name         shop_city  shop_tier  bm25_score
    19577930                                                                      Kopi solong/kopi Aceh 1000 grm                                                                                                   https://www.tokopedia.com/widarstore/kopi-solongkopi-aceh-1000-grm?extParam=ivf%3Dfalse%26keyword%3Dkopi+khas%26search_id%3D20260122105211577BCC914F9E5523874Z%26src%3Dsearch     makana

***newest code (dengan pertimbangan query lebih dari 3 kosa kata)***

In [ ]:
# =========================================================
# 1. IMPORT LIBRARY
# =========================================================
import re
import pickle
import pandas as pd
from rank_bm25 import BM25Okapi


# =========================================================
# 2. KONFIGURASI
# =========================================================
CSV_PATH = "../olahData/nondup_labeled_dataset.csv"
ENCODING = "utf-8"
TOPK = 20

TEXT_COLS = [
    "name_clean", 
    "category_clean", 
    "city_clean"
]

OUT_COLS = [
    "id", "name", "url",
    "category_breadcrumb",
    "price_number",          
    "discountPercentage",
    "ratingAverage", "shop_id",
    "shop_name", "shop_city", "shop_tier",
    "countSold", "has_Promo", "umkm_label",
]


# =========================================================
# 3. LOAD DATA
# =========================================================
df = pd.read_csv(CSV_PATH, encoding=ENCODING)

# Rename kolom jika diperlukan
rename_map = {
    "shop.id": "shop_id",
    "shop.name": "shop_name",
    "shop.city": "shop_city",
    "shop.tier": "shop_tier",
    "category.breadcrumb": "category_breadcrumb",
    "price.number": "price_number",
    "price.discountPercentage": "discountPercentage",
}

for old, new in rename_map.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

# Fallback jika kolom clean belum tersedia
if "name_clean" not in df.columns:
    df["name_clean"] = df["name"].fillna("").astype(str).str.lower()

if "category_clean" not in df.columns:
    df["category_clean"] = df["category_breadcrumb"].fillna("").astype(str).str.lower()

if "city_clean" not in df.columns:
    df["city_clean"] = df["shop_city"].fillna("").astype(str).str.lower()


# =========================================================
# 4. UTIL FUNCTIONS
# =========================================================
def safe_get_col(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df))


def basic_tokens(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []


def tokenize_with_ngrams(text: str, max_n: int = 3):
    """
    Tokenisasi Unigram + Bigram + Trigram
    """
    toks = basic_tokens(text)
    all_tokens = toks.copy()

    for n in range(2, max_n + 1):
        ngrams = [
            "_".join(toks[i:i+n])
            for i in range(len(toks) - n + 1)
        ]
        all_tokens.extend(ngrams)

    return all_tokens


def build_document(df: pd.DataFrame) -> pd.Series:
    parts = [safe_get_col(df, c) for c in TEXT_COLS]
    doc = parts[0]
    for p in parts[1:]:
        doc = doc + " " + p
    return doc


def normalize_minmax(series):
    series = series.fillna(0).astype(float)
    if series.max() == series.min():
        return pd.Series([0] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min())


# =========================================================
# 5. BUILD CORPUS & BM25 MODEL
# =========================================================
print("Membangun indeks BM25...")

df["doc"] = build_document(df)
corpus_tokens = df["doc"].apply(
    lambda x: tokenize_with_ngrams(x, max_n=3)
).tolist()

bm25 = BM25Okapi(corpus_tokens, k1=1.5, b=0.75)


# =========================================================
# 6. SEARCH FUNCTIONS
# =========================================================
def bm25_search(query: str, topk: int = 20, require_all_terms: bool = False):
    """
    Fungsi pencarian BM25 dengan opsi AND-filter untuk query panjang
    """
    q_tokens = tokenize_with_ngrams(query, max_n=3)
    q_unigrams = basic_tokens(query)

    scores = bm25.get_scores(q_tokens)

    df_out = df.copy()
    df_out["bm25_score"] = scores
    df_out["bm25_norm"] = normalize_minmax(df_out["bm25_score"])

    # AND Filter (opsional untuk query panjang)
    if require_all_terms and q_unigrams:
        doc_tokens = df_out["doc"].apply(lambda x: set(basic_tokens(x)))
        mask = doc_tokens.apply(lambda s: all(t in s for t in q_unigrams))
        df_out = df_out[mask]

    df_out = df_out.sort_values("bm25_score", ascending=False).head(topk)

    cols = [c for c in OUT_COLS if c in df_out.columns]
    cols = cols + ["bm25_score", "bm25_norm"]

    return df_out[cols].reset_index(drop=True)


def bm25_candidates(query: str, top_n: int = 2000):
    """
    Candidate retrieval untuk Hybrid Ranking (Top-N ±2% dataset)
    """
    q_tokens = tokenize_with_ngrams(query, max_n=3)
    scores = bm25.get_scores(q_tokens)

    df_out = df.copy()
    df_out["bm25_score"] = scores
    df_out["bm25_norm"] = normalize_minmax(df_out["bm25_score"])

    return df_out.sort_values("bm25_score", ascending=False).head(top_n)


# =========================================================
# 7. MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    print("BM25 siap digunakan.")
    q = input("Masukkan query: ").strip()

    result = bm25_search(
        q,
        topk=TOPK,
        require_all_terms=len(q.split()) >= 3
    )

    print("\nHasil Pencarian:")
    print(result.to_string(index=False))

    result.to_csv("bm25_results5.csv", index=False, encoding="utf-8-sig")
    print("\nDisimpan: bm25_results5.csv")

Membangun indeks BM25...
BM25 siap digunakan.

Hasil Pencarian:
          id                                                                                                                                                                                                                                                          name                                                                                                                                                                                                                                                                                                                                                                                                                                                     url                                    category_breadcrumb  price_number  discountPercentage  ratingAverage             shop_id                      shop_name          shop_city umkm_label  bm25_score  bm25_norm
100398047155         

In [2]:
import re
import pandas as pd
from rank_bm25 import BM25Okapi

# ========= KONFIG =========
CSV_PATH = "../output/aksesoriWanita_enriched.csv"   # ganti sesuai file kamu
ENCODING = "utf-8"                         # kalau error coba "utf-8-sig"
TOPK = 20

# Kolom yang dipakai untuk membentuk dokumen (boleh kamu ubah)
TEXT_COLS = [
    "name",
    "category_breadcrumb",   # atau "categoryBreadcrumbs" / "category.breadcrumb"
    "shop_city"              # atau "shop.city"
]

# Kolom output yang mau ditampilkan
OUT_COLS = [
    "id", "name", "url",
    "category_breadcrumb",
    "price_number",
    "rating",
    "discountPercentage",
    "shop_id", "shop_name", "shop_city", "shop_tier"
]

# ========= UTIL =========
def safe_get_col(df: pd.DataFrame, col: str) -> pd.Series:
    """Ambil kolom kalau ada, kalau tidak ada buat kolom kosong."""
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df))

def tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def build_document(df: pd.DataFrame) -> pd.Series:
    parts = []
    for c in TEXT_COLS:
        parts.append(safe_get_col(df, c))
    doc = parts[0]
    for p in parts[1:]:
        doc = doc + " " + p
    return doc

# ========= LOAD =========
df = pd.read_csv(CSV_PATH, encoding=ENCODING)

# Pastikan kolom penting ada (kalau berbeda nama, mapping manual di bawah)
# Kalau CSV kamu pakai nama kolom versi nested (mis. "shop.id"), kamu bisa rename.
rename_map = {
    # contoh mapping yang sering terjadi (aktifkan bila perlu):
    "shop.id": "shop_id",
    "shop.name": "shop_name",
    "shop.city": "shop_city",
    "shop.tier": "shop_tier",
    "category.breadcrumb": "category_breadcrumb",
    "price.number": "price_number",
    "price.discountPercentage": "discountPercentage",
    "mediaURL.image": "mediaURL_image",
}
for old, new in rename_map.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

# ========= BUILD CORPUS =========
df["doc"] = build_document(df)
corpus_tokens = df["doc"].apply(tokenize).tolist()

bm25 = BM25Okapi(corpus_tokens)

# ========= SEARCH =========
def bm25_search(query: str, topk: int = 20) -> pd.DataFrame:
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)

    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:topk]
    out = df.iloc[top_idx].copy()
    out["bm25_score"] = [scores[i] for i in top_idx]

    # pilih kolom output yang ada saja
    cols = [c for c in OUT_COLS if c in out.columns]
    cols = cols + ["bm25_score"]
    return out[cols].reset_index(drop=True)

if __name__ == "__main__":
    print("BM25 siap. Contoh query:")
    q = input("Masukkan query (contoh: 'baju wanita korea'): ").strip()
    res = bm25_search(q, topk=TOPK)
    print(res.to_string(index=False))

    res.to_csv("bm25_results.csv", index=False, encoding="utf-8-sig")
    print("\n Disimpan: bm25_results.csv")


BM25 siap. Contoh query:
          id                                                                                                                                                                 name                                                                                                                                                                                                                                                                                                                                                    url                           category_breadcrumb  price_number  discountPercentage             shop_id          shop_name         shop_city  shop_tier  bm25_score
100643939905                                                                                                                  Gelang Titanium Dengan Liontin Cantik Gelang Cantik                                                                                                                     

In [5]:
import argparse
import glob
import math
import itertools
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from rank_bm25 import BM25Okapi

# ---------- utils ----------
def basic_tokens(text: str):
    if not isinstance(text, str): text = ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []

def load_corpus(path, text_cols):
    df = pd.read_csv(path, encoding="utf-8")
    if "id" in df.columns:
        df["id"] = df["id"].astype(str)
    else:
        df["id"] = df.index.astype(str)
    parts = [
        df[c].fillna("").astype(str) if c in df.columns else pd.Series([""] * len(df))
        for c in text_cols
    ]
    doc = parts[0]
    for p in parts[1:]:
        doc = doc + " " + p
    df["doc"] = doc
    tokens = df["doc"].apply(basic_tokens).tolist()
    return df, tokens

def average_precision_at_k(ranked_ids, relevant_set, k):
    if not relevant_set:
        return 0.0
    hits = 0
    sum_prec = 0.0
    for i, doc in enumerate(ranked_ids[:k], start=1):
        if doc in relevant_set:
            hits += 1
            sum_prec += hits / i
    return sum_prec / len(relevant_set)

def dcg_at_k(rels, k):
    return sum((2 ** r - 1) / math.log2(i + 1 + 1) for i, r in enumerate(rels[:k]))

def ndcg_at_k(ranked_ids, rels_map, k):
    rels = [rels_map.get(d, 0) for d in ranked_ids[:k]]
    dcg = dcg_at_k(rels, k)
    ideal = sorted(rels_map.values(), reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_bm25(df, corpus_tokens, qrels_df, k1, b, topk=10):
    bm25 = BM25Okapi(corpus_tokens, k1=k1, b=b)
    queries = qrels_df["query"].unique()
    ap_list, ndcg_list = [], []
    for q in queries:
        q_tokens = basic_tokens(q)
        if not q_tokens:
            continue
        scores = bm25.get_scores(q_tokens)
        top_idx = np.argsort(scores)[::-1][:topk]
        ranked_ids = df.iloc[top_idx]["id"].astype(str).tolist()
        sub = qrels_df[qrels_df["query"] == q]
        rel_map = dict(zip(sub["doc_id"].astype(str), sub["relevance"].astype(int)))
        relevant_set = {d for d, r in rel_map.items() if r > 0}
        ap = average_precision_at_k(ranked_ids, relevant_set, topk)
        ndcg = ndcg_at_k(ranked_ids, rel_map, topk)
        ap_list.append(ap)
        ndcg_list.append(ndcg)
    return np.mean(ap_list) if ap_list else 0.0, np.mean(ndcg_list) if ndcg_list else 0.0

def grid_search(df, corpus_tokens, qrels_df, k1_range, b_range, topk=10):
    rows = []
    for k1, b in itertools.product(k1_range, b_range):
        map_k, ndcg_k = evaluate_bm25(df, corpus_tokens, qrels_df, k1, b, topk=topk)
        rows.append({"k1": float(k1), "b": float(b), "MAP@k": float(map_k), "NDCG@k": float(ndcg_k)})
        print(f"k1={k1:.3f} b={b:.3f} => MAP@{topk}={map_k:.4f} NDCG@{topk}={ndcg_k:.4f}")
    return pd.DataFrame(rows)

# ---------- main ----------
def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--corpus", default=None, help="path to corpus csv (default: try ../output/*_enriched.csv)")
    p.add_argument("--qrels", default=None, help="path to qrels csv (cols: query,doc_id,relevance)")
    p.add_argument("--topk", type=int, default=10)
    p.add_argument("--text-cols", default="name,category_breadcrumb,shop_city",
                   help="comma separated columns to build doc (default: name,category_breadcrumb,shop_city)")
    p.add_argument("--coarse-k1", default="0.5:2.0:0.25",
                   help="coarse k1 range start:end:step")
    p.add_argument("--coarse-b", default="0.0:1.0:0.1",
                   help="coarse b range start:end:step")
    args = p.parse_args(argv)

    # find corpus if not provided
    if args.corpus is None:
        search = os.path.join(os.path.dirname(__file__), "..", "output", "*_enriched.csv")
        found = sorted(glob.glob(os.path.normpath(search)))
        if not found:
            raise FileNotFoundError(f"No *_enriched.csv found under ../output. Check {os.path.join(os.path.dirname(__file__), '..', 'output')}")
        # prefer aksesoriWanita if present
        default_choice = next((f for f in found if "aksesoriWanita" in os.path.basename(f)), found[0])
        args.corpus = os.path.normpath(default_choice)
        print(f"No --corpus given. Using detected corpus: {args.corpus}")

    if not os.path.exists(args.corpus):
        raise FileNotFoundError(f"Corpus not found: {args.corpus}")

    # qrels required for tuning
    if args.qrels is None or not os.path.exists(args.qrels):
        raise FileNotFoundError("Qrels file required for tuning (--qrels). Provide CSV with columns: query,doc_id,relevance")

    # parse ranges
    def parse_range(spec):
        s, e, step = map(float, spec.split(":"))
        return np.round(np.arange(s, e + 1e-9, step), 6)

    k1_coarse = parse_range(args.coarse_k1)
    b_coarse = parse_range(args.coarse_b)

    TEXT_COLS = [c.strip() for c in args.text_cols.split(",")]

    print("Loading corpus:", args.corpus)
    df, corpus_tokens = load_corpus(args.corpus, TEXT_COLS)
    print("Docs:", len(df))

    qrels = pd.read_csv(args.qrels, encoding="utf-8")
    required = {"query", "doc_id", "relevance"}
    if not required.issubset(set(qrels.columns)):
        raise ValueError(f"Qrels missing required columns. Found: {qrels.columns.tolist()} Expected: {required}")
    qrels["doc_id"] = qrels["doc_id"].astype(str)

    # coarse search
    print("Starting coarse grid search...")
    coarse = grid_search(df, corpus_tokens, qrels, k1_coarse, b_coarse, topk=args.topk)
    coarse.to_csv("bm25_grid_coarse.csv", index=False)
    best = coarse.sort_values("MAP@k", ascending=False).iloc[0]
    best_k1, best_b = float(best["k1"]), float(best["b"])
    print("Best coarse:", best_k1, best_b)

    # fine search around best
    k1_fine = np.round(np.arange(max(0.1, best_k1 - 0.2), best_k1 + 0.2001, 0.05), 6)
    b_fine = np.round(np.arange(max(0.0, best_b - 0.1), min(1.0, best_b + 0.1001), 0.02), 6)

    print("Starting fine grid search...")
    fine = grid_search(df, corpus_tokens, qrels, k1_fine, b_fine, topk=args.topk)
    fine.to_csv("bm25_grid_fine.csv", index=False)

    best_final = pd.concat([coarse, fine]).sort_values("MAP@k", ascending=False).iloc[0]
    print("Best final params:", best_final.to_dict())

    # heatmap (MAP)
    try:
        pivot = fine.pivot(index="k1", columns="b", values="MAP@k")
        plt.figure(figsize=(8, 6))
        plt.title("MAP heatmap (fine grid)")
        plt.imshow(pivot.values, origin="lower", aspect="auto", cmap="viridis")
        plt.xticks(range(len(pivot.columns)), [f"{c:.2f}" for c in pivot.columns], rotation=90)
        plt.yticks(range(len(pivot.index)), [f"{i:.2f}" for i in pivot.index])
        plt.colorbar(label="MAP@k")
        plt.tight_layout()
        plt.savefig("bm25_map_heatmap.png", dpi=150)
        print("Saved bm25_map_heatmap.png")
    except Exception as e:
        print("Couldn't produce heatmap:", e)

if __name__ == "__main__":
    # handle notebook execution where argparse sees ipykernel args
    try:
        main(sys.argv[1:])
    except SystemExit:
        main([])

usage: ipykernel_launcher.py [-h] [--corpus CORPUS] [--qrels QRELS]
                             [--topk TOPK] [--text-cols TEXT_COLS]
                             [--coarse-k1 COARSE_K1] [--coarse-b COARSE_B]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Annisa\AppData\Roaming\jupyter\runtime\kernel-v351154fda6308086edd55af28472799331cc57550.json


NameError: name '__file__' is not defined